In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_astradb import AstraDBVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [18]:
## load text file
loader=TextLoader("langchain_rag_dataset.txt")
raw_docs=loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks


[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [ ]:

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)




ASTRA_DB_API_ENDPOINT = ""
ASTRA_DB_APPLICATION_TOKEN = ""



vector_store = AstraDBVectorStore(
    collection_name="MMR",
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
)


llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)


# vector_store.add_documents(documents=chunks)  -> alreay loaded


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4094.25it/s]


In [20]:
#lambda_mult -> can also add this for like 70% reveland doc and 30% diverese

### Step 3: Create MMR Retirever
retriever=vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [21]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")


In [22]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [23]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents and memory in several ways:

1. **Agents**: LangChain allows LLMs to act as agents that can decide which tool to call and in what order during a task. This flexibility enables agents to make decisions based on the context and requirements of the task.

2. **Memory**: LangChain helps models retain previous interactions through its conversational memory mechanisms, like ConversationBufferMemory and ConversationSummaryMemory. This makes multi-turn conversations more coherent by allowing the agent to remember prior interactions and context, which is crucial for maintaining a coherent flow of conversation over multiple turns.

3. **Tools**: Agents can utilize a variety of tools, such as calculators, search APIs, or custom functions, based on the instructions they receive. This adaptability enhances the capabilities of the agents and allows them to perform a wide range of tasks effectively.

4. **APIs and Databases**: Agents can interact with external API

In [24]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='30d53730f2bd43e9988647730de9bdac', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='4a5a066de35b454cb5cd689179feafa2', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.'),
  Document(id='e52fc66cd40b4d4e8312cecdee322642', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applica